# TRIDENT T69 ? approved hybrid_v1_dense4 validation benchmark
Run on **Colab ? Runtime ? Change runtime type ? T4 GPU**, then Run all.
Upload the three files supplied alongside this notebook to `MyDrive/TRIDENT-T69/hybrid_validation_inputs`.
This notebook runs inference only: 123 validation frames, fixed configuration, no training, no parameter tuning, no official test sample payloads.
It compares bicubic, B0, B0 + classical, B0 + geometry and complete hybrid, with an additional consistency-only control.
Each execution uses a fresh results directory on Drive. Never reuse a failed directory; retain it as evidence.
The source ZIP was freshly built from the current working tree, including the new hybrid modules. Its SHA256 below pins the actual bytes, including uncommitted additions; no older experiment source ZIP is used.

## 1. Mount Drive and select input paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, hashlib, zipfile, subprocess, uuid, datetime
from pathlib import Path
INPUT = Path('/content/drive/MyDrive/TRIDENT-T69/hybrid_validation_inputs')
SOURCE_ZIP = INPUT / 'trident_hybrid_current_source.zip'
VALIDATION_ZIP = INPUT / 'nyu_validation123_only.zip'
CHECKPOINT = INPUT / 'B0_epoch_002.pt'
RESULTS_ROOT = Path('/content/drive/MyDrive/TRIDENT-T69/verification_results/hybrid_v1_dense4')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:10]
WORK = Path('/content') / ('trident_hybrid_' + RUN_ID)
REPO, DATA = WORK / 'repo', WORK / 'validation_data'
OUTPUT = RESULTS_ROOT / RUN_ID
PY = sys.executable
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print('New output:', OUTPUT)

## 2. Require CUDA/Tesla T4 and verify all inputs before loading the model

In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA unavailable. Select a T4 GPU runtime.'
assert torch.cuda.device_count() == 1, 'This benchmark expects one visible GPU.'
assert torch.cuda.get_device_name(0) == 'Tesla T4', 'A Tesla T4 is required; do not substitute another device.'
print('Python:', sys.version, '\nTorch:', torch.__version__, '\nCUDA:', torch.version.cuda,
      '\nGPU:', torch.cuda.get_device_name(0), '\nVRAM:', torch.cuda.get_device_properties(0).total_memory)
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
EXPECTED = {SOURCE_ZIP: 'f08b6ccca6fde3b040245a048556d6669a7cfe1b28f76267c649f6ce6fd07e06',
             VALIDATION_ZIP: '9daad64f7641969b8ee515cf5062dcda83339f2d7850fcf2fc6f20104c517267',
             CHECKPOINT: 'a6416c47a16ec78e97c8b734ae7fb23cba4dbbeb5f5564a3e02d61a4a159c1b2'}
for path, expected in EXPECTED.items():
    assert path.is_file(), f'Missing input: {path}'
    actual = digest(path)
    assert actual == expected, f'SHA256 mismatch: {path}'
    print(path.name, actual)
assert not WORK.exists() and not OUTPUT.exists(), 'Refusing to overwrite an existing run.'

## 3. Stage the verified current source and validation-only data
The mixed manifest contains split labels; only validation payloads are present in the supplied data ZIP.

In [ ]:
from pathlib import PurePosixPath
WORK.mkdir()
REPO.mkdir()
DATA.mkdir()
def safe_members(z):
    names = z.namelist()
    assert len(names) == len(set(names)), 'Duplicate archive member'
    for item in z.infolist():
        p = PurePosixPath(item.filename)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in item.filename
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink member forbidden'
    return set(names)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    safe_members(z)
    z.extractall(REPO)
with zipfile.ZipFile(VALIDATION_ZIP) as z:
    names = safe_members(z)
    manifest_bytes = z.read('manifest_development.json')
    assert hashlib.sha256(manifest_bytes).hexdigest() == 'd1cdbcf0ff3ee96f2e0c173c3c4701ed3f5c526ab517ce3b988268c3c6a391f5'
    rows = [r for r in json.loads(manifest_bytes) if r['split'] == 'validation']
    reference = json.loads((REPO/'docs/pilot_dense4/per_frame.json').read_text())
    assert len(rows) == 123 and [r['id'] for r in rows] == [r['id'] for r in reference]
    allowed = {'manifest_development.json'} | {r[k] for r in rows for k in ('rgb','depth','mask')}
    assert names == allowed and len(names) == 370, 'Archive must contain ONLY validation payloads and manifest'
    z.extractall(DATA)
for row, frame in zip(rows, reference):
    for key in ('rgb','depth','mask'):
        assert digest(DATA/row[key]) == frame['source_files_sha256'][key]
assert digest(REPO/'configs/hybrid_v1_dense4.json') == 'a2b943d2055af24504a40c1dbcee320f68977908351b6ce297fb84e58aeb4f7c'
os.chdir(REPO)
print('Verified 123 frames; no training/test sample files were staged.')

## 4. Install additional dependencies without changing Colab PyTorch; run the complete test suite

In [ ]:
# Use Colab's existing runtime directly: no venv or ensurepip.
import tempfile
PY = sys.executable
print("Colab Python:", PY)
torch_probe = """
import json, torch
print(json.dumps(dict(version=str(torch.__version__), cuda=torch.version.cuda,
 location=torch.__file__, cuda_available=torch.cuda.is_available())))
"""
TORCH_BEFORE = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_BEFORE["cuda_available"], "Select a GPU runtime; Colab CUDA PyTorch must already be available"

# Resolve only the additional requirements first. Reject any plan that would
# touch PyTorch or its CUDA stack, then install exactly that plan without resolving
# further dependencies. This cannot pull in a CPU-only torch wheel.
with tempfile.TemporaryDirectory(prefix="trident-pip-") as temporary:
    report = Path(temporary) / "install-plan.json"
    subprocess.run([str(PY), "-m", "pip", "install", "--dry-run",
                    "--only-binary=:all:", "--report", str(report),
                    "-r", str(REPO / "requirements-colab.txt")], check=True)
    plan = json.loads(report.read_text())["install"]
    protected = {"torch", "torchvision", "torchaudio", "torchtext", "triton", "pytorch-triton"}
    for item in plan:
        name = item["metadata"]["name"].lower().replace("_", "-")
        assert name not in protected and not name.startswith("nvidia-"), f"Refusing to replace Colab GPU package: {name}"
    urls = [item["download_info"]["url"] for item in plan]
    if urls:
        subprocess.run([str(PY), "-m", "pip", "install", "--no-deps",
                        "--only-binary=:all:", *urls], check=True)
subprocess.run([str(PY), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
TORCH_AFTER = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_AFTER == TORCH_BEFORE, "Colab PyTorch/CUDA changed during setup"
probe = """
import json, platform, torch, numpy, scipy, cv2, PIL, h5py
assert (2, 6) <= tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (3, 0)
print(json.dumps(dict(python=platform.python_version(), torch=str(torch.__version__),
 cuda=torch.version.cuda, numpy=numpy.__version__, scipy=scipy.__version__,
 opencv=cv2.__version__, pillow=PIL.__version__, h5py=h5py.__version__,
 gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
 gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)))
"""
ENVIRONMENT = json.loads(subprocess.check_output([str(PY), "-c", probe], text=True))
print(json.dumps(ENVIRONMENT, indent=2))
assert ENVIRONMENT["gpu"], "No CUDA GPU available: select a GPU runtime. Do not run the pilot on CPU."
# Complete project suite after installation; CPU tests do not train a model.
subprocess.run([str(PY), "-m", "pytest", "-q"], check=True,
               env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))

## 5. Execute the fixed validation-only evaluator
Already approved: one frozen benchmark. One warm-up and three timing repetitions per frame. Predictions are saved. A failure stops this notebook and preserves the failed output. No automatic retry or tuning is performed.

In [ ]:
assert torch.cuda.is_available() and torch.cuda.get_device_name(0) == 'Tesla T4'
for path, expected in EXPECTED.items():
    assert digest(path) == expected, f'Input changed: {path}'
assert not OUTPUT.exists(), 'Never overwrite benchmark evidence'
command = [PY, '-B', '-m', 'trident_depth.evaluation.hybrid_v1_dense4',
           '--execute-validation', '--manifest', str(DATA/'manifest_development.json'),
           '--checkpoint', str(CHECKPOINT), '--output', str(OUTPUT),
           '--config', str(REPO/'configs/hybrid_v1_dense4.json'),
           '--reference', str(REPO/'docs/pilot_dense4/per_frame.json'),
           '--device', 'cuda', '--warmup', '1', '--repeats', '3', '--save-predictions']
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
launcher = RESULTS_ROOT/(RUN_ID+'_launcher.json')
with launcher.open('x') as f:
    json.dump(dict(command=command, inputs={str(p):h for p,h in EXPECTED.items()},
                   output=str(OUTPUT), gpu=torch.cuda.get_device_name(0),
                   optimizer_steps=0, validation_frames=123), f, indent=2)
log_path = RESULTS_ROOT/(RUN_ID+'_console.log')
with log_path.open('x') as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    returncode = process.wait()
assert returncode == 0, f'Benchmark failed; preserve {OUTPUT} and {log_path}. Do not tune or reuse output.'

## 6. Verify completion, isolation and unchanged inputs; display reports

In [ ]:
report = json.loads((OUTPUT/'report.json').read_text())
access = json.loads((OUTPUT/'access_audit.json').read_text())
print(json.dumps(report, indent=2))
print(json.dumps(access['summary'], indent=2))
assert access['summary']['validation_frames'] == 123
assert access['summary']['official_test_sample_files_opened'] == 0
assert access['summary']['training_sample_files_opened'] == 0
assert not access['summary']['forbidden_access_attempts']
assert len(list((OUTPUT/'predictions').glob('*.npz'))) == 123
assert report['status'] == 'COMPLETE', 'Baseline replay or benchmark failed'
# The evaluator writes the complete access ledger; report includes the isolation summary.
for path, expected in EXPECTED.items():
    assert digest(path) == expected, f'Original input changed: {path}'
from IPython.display import display, Markdown
display(Markdown((OUTPUT/'report.md').read_text()))
print('Persistent results:', OUTPUT)
print('Contains predictions/, per_frame.json, report.json, report.md, provenance.json, config.json, access_audit.json.')
print('No training or export was performed. Stop here.')